## Test notebook for auto bining

In [12]:
import os
import pandas as pd
import numpy as np

In [13]:
#read data from data/PKV RED REMARKS.xlsm
path = os.path.join('..', 'data', 'PKV RED REMARKS.xlsm')
print(path)
data = pd.read_excel(path, sheet_name='Sorted Issues')
bins = pd.read_excel(path, sheet_name='General Bin List')
print("Data loaded successfully")

..\data\PKV RED REMARKS.xlsm
Data loaded successfully


In [14]:
bins_dict = {col: bins[col].dropna().tolist() for col in bins.columns}
print(bins_dict)
print(bins_dict.keys())

{'Unassigned': [], 'As Intended': [], '12V system problem': ['12V drained', 'Battery cable', 'CEM B11D887 LIN BMS', 'Critical charging message', 'DIM message 12v system service required', 'Low Voltage system message', 'Message: 12V battery cannot be charged', 'Multiple warning messages', 'Redundant battery front, low SOC', 'Steering wheel button, light on', 'Support battery'], '12V system problem - WISE': [], '3:rd party APP': ['Air quality app', 'Amazon Prime App', 'App-drive mode availability', 'Apple Music', 'Apps not available?', 'AudioWagon', 'B&W Experience app', 'Dolby atmos ', 'EasyPark app', 'Game app', 'Google play issue', 'HBO App', 'Himalaya App', 'Libby app', 'MAX', 'Missing 3:rd party APP', 'QQ music', 'Spotify app', 'Spotify not possible to play/pause', 'SR app', 'Storytel app', 'Tidal app', 'Tunein app', 'Vivaldi app', 'Waze app'], 'AC': ['Noise', 'Other', 'Performance', 'System leakage', 'Vibrations'], 'AC charge issue': [], 'ACC': ['ACC - Driver assistance system book

In [15]:
import numpy as np
from sklearn.model_selection import train_test_split
#create new df from "data" containing only columns "Summary" and "Solution"
filtered_data = data[data['BIN'].isin(list(bins_dict.keys())[1:])]
# features = filtered_data[['Summary', 'Solution']]
# target = filtered_data[['BIN', 'BIN Specific']]
# The original 'data' remains unchanged after this cell

print(filtered_data.shape)
# Shuffle the rows
filtered_data = filtered_data.sample(frac=1, random_state=42).reset_index(drop=True)

# Split into train and test sets (e.g., 80% train, 20% test)
train_data, test_data = train_test_split(filtered_data, test_size=0.15, random_state=42)

features_train = train_data[['Summary', 'Solution']]
target_train = train_data[['BIN', 'BIN Specific']]
features_test = test_data[['Summary', 'Solution']]
target_test = test_data[['BIN', 'BIN Specific']]

(2571, 19)


In [16]:
print(features_train.head())
print(target_train.head())

                                                Summary  \
1711  AGX28J/XLP1177 - Not possible to play any radi...   
332              EBJ47P/XLP1174 - GPS position is wrong   
432    RSI suggesting 80km/h several times where the...   
407         ADAS telltale shown after parking manoveour   
508   After using front defrost - temperature settin...   

                     Solution  
1711  Safe Vehicle Automation  
332      Connected Experience  
432                    ADADAS  
407                    ADADAS  
508       Vehicle Engineering  
                                     BIN           BIN Specific
1711  Electronic Stability Control (ESC)            DIM Message
332            Navigation System Problem   Not correct position
432                                  RSI  RSI Not correct speed
407         General Intellisafe Problems          ADAS Telltale
508                      General climate                  Logic


In [17]:
OLD_BIN_ALIASES: dict[str, str] = {
    "acc": "ACC",
    "adas": "General Intellisafe Problems",
    "ahbc": "Automatic high beam",
    "alarm": "Alarm",
    "app": "Mobile App problem",
    # "battery & charging": "No charging",
    "blis": "BLIS",
    "bluetooth": "Bluetooth",
    # "brake lights": "Rear lamps & lighting",
    "climate": "General climate",
    # "collision avoidance": "Collision Avoidance",
    "csa": "Curve Speed Assist",
    "csd": "CSD",
    # "csd & dim": "CSD and DIM black",
    # "digital key": "Key Not Found",
    "dim": "DIM",
    # "dim messages": "DIM",
    "display": "Display low brightness/dark",
    "dms": "Driver Monitoring System (DMS)",
    "doors & locks": "Door lock",
    # "driveline": "Driveline issue",
    "esc": "Electronic Stability Control (ESC)",
    # "factory reset": "Driver profile and personal settings",
    # "google": "Navigation System Problem",
    # "google assistant": "Gemini Voice assistant",
    # "google maps": "Navigation System Problem",
    # "gps": "Navigation System Problem",
    # "graphical bug": "Graphic bugs",
    # "hardware": "Other",
    # "hazard warning": "DIM",
    "hod": "Hands off Detection (HOD) system",
    "isa": "ISA",
    # "key fob": "Key Fob HW related",
    "lca": "LCA - Lane Change Assist",
    "ldw": "LDW",
    # "lights": "Headlamps",
    "lka": "LKA",
    # "media & sound": "Audio Sound Problem",
    "mirrors": "Outer rear view mirrors",
    "navigation": "Navigation System Problem",
    "nvh": "Driveline vibrations",
    "opr": "OPR",
    "other": "Other",
    "pa": "Pilot Assist",
    "pac": "PAC 360",
    # "phone": "Mobile App problem",
    # "pilot assist": "Pilot Assist",
    "pot": "POT",
    "profile": "Driver profile and personal settings",
    # "radio": "Radio problem",
    # "rain sensor": "Rain sensor",
    "rsi": "RSI",
    "seatbelt": "Seatbelt front",
    "seats": "Seat front",
    "sla": "Slippery Road",
    "soc": "Range info",
    "sos/e-call": "E-call issues",
    # "spotify": "Audio Sound Problem",
    # "steering wheel": "Steering wheel",
    # "suspension": "Suspension",
    "tpms": "TPMS",
    # "unassigned": "Other",
    # "unknown": "Other",
    # "as intended": "Other",
    # "n/a": "Other",
    # "conceptual": "Other",
    # "widget": "Driver & Vehicle Info",
    # "windows & wipers": "Front wipers",
    "wpc": "Wireless Phone Charger",
}

def replace_words_in_text(text, mapping):
    words = text.lower().split()
    replaced = [mapping.get(word, word) for word in words]
    return ' '.join(replaced)

def formated_features(features):
    features = features.fillna('empty')
    features['Summary'] = features['Summary'].apply(lambda x: replace_words_in_text(x, OLD_BIN_ALIASES))
    # print(features['Summary'].iloc[150:200])
    # features = features.astype(str).str.lower()
    features = features['Solution'].astype(str) + ' | ' + features['Summary'].astype(str)
    features = features.str.lower()
    return features

def formated_target(target):
    target = target.fillna('')
    target = target['BIN'].astype(str) + ' | ' + target['BIN Specific'].astype(str)
    return target

features_train = formated_features(features_train)
target_train = formated_target(target_train)
features_test = formated_features(features_test)
target_test = formated_target(target_test)

# print(features.iloc[100:150])
# print(target.head())
# print(features.shape, target.shape)

### TF-IDF + KNN

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
mat_features = vectorizer.fit_transform(features_train)

print(vectorizer.get_feature_names_out())
print(mat_features.toarray())
print(mat_features.toarray().shape)



['10' '100' '110' ... 'zcra' 'zone' 'är']
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
(2185, 1956)


In [19]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(
    n_neighbors=5,
    metric="cosine",
    weights="distance"
)
# knn.fit(mat_features, target_train['BIN'])
knn.fit(mat_features, target_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'distance'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [20]:
for i in range(30, 60):
    sample = features_test.iloc[i]
    sample_vector = vectorizer.transform([sample])
    distances, indices = knn.kneighbors(sample_vector)
    distances = 1 - distances

    print("Sample: ", sample)
    # print("Predicted BIN:", target_train['BIN'].iloc[indices[0][0]])
    # print("Actual BIN:", target_test['BIN'].iloc[i])
    print("Predicted BIN (k=1)", "(d=)", round(distances[0][0], 2), ":", target_train.iloc[indices[0][0]])
    print("Predicted BIN (k=2)", "(d=)", round(distances[0][1], 2), ":", target_train.iloc[indices[0][1]])
    print("Predicted BIN (k=3)", "(d=)", round(distances[0][2], 2), ":", target_train.iloc[indices[0][2]])
    print("Predicted BIN (k=4)", "(d=)", round(distances[0][3], 2), ":", target_train.iloc[indices[0][3]])
    print("Predicted BIN (k=5)", "(d=)", round(distances[0][4], 2), ":", target_train.iloc[indices[0][4]])
    print("Actual BIN:", target_test.iloc[i])
    print()

Sample:  vehicle engineering | unlocked car not possible to open
Predicted BIN (k=1) (d=) 0.64 : Door handle | Hard to open
Predicted BIN (k=2) (d=) 0.59 : Door handle | Hard to open
Predicted BIN (k=3) (d=) 0.53 : Door handle | Hard to open
Predicted BIN (k=4) (d=) 0.52 : Door handle | Hard to open
Predicted BIN (k=5) (d=) 0.52 : CSD partly function | Grey tiles appeared in CSD 
Actual BIN: Lock/unlock doors | Approach/Leave with phone

Sample:  vehicle engineering | could not adjust rhs mirror at first try
Predicted BIN (k=1) (d=) 0.52 : Lock/unlock doors | Approach/Leave with keyfob
Predicted BIN (k=2) (d=) 0.42 : Steering column | Diff to adjust
Predicted BIN (k=3) (d=) 0.4 : Door handle | Hard to open
Predicted BIN (k=4) (d=) 0.4 : Outer rear view mirrors | Not working
Predicted BIN (k=5) (d=) 0.39 : CSD partly function | Grey tiles appeared in CSD 
Actual BIN: Outer rear view mirrors | Not working

Sample:  vehicle engineering | false pinch protection
Predicted BIN (k=1) (d=) 0.7

### Performance

In [21]:

from collections import Counter

total_predictions = features_test.shape[0]
k1_correct_predictions = 0
k3_correct_predictions = 0
k5_correct_predictions = 0
correct_dist_k1 = []
correct_dist_k3 = []
correct_dist_k5 = []
incorrect_dist_k1 = []
incorrect_dist_k3 = []
incorrect_dist_k5 = []

def find_most_common(els):
    counter = Counter(els)
    max_count = max(counter.values())
    for el in els:
        if counter[el] == max_count:
            return el

for i in range(features_test.shape[0]):
    sample = features_test.iloc[i]
    sample_vector = vectorizer.transform([sample])
    distances, indices = knn.kneighbors(sample_vector)
    distances = 1 - distances

    #k=1
    if target_train.iloc[indices[0][0]] == target_test.iloc[i]:
        k1_correct_predictions += 1
        correct_dist_k1.append(np.mean(distances[0][0]))
    else:
        incorrect_dist_k1.append(np.mean(distances[0][0]))

    # #k=2
    # if find_most_common(target_train.iloc[indices[0][0:2]]) == target_test.iloc[i]:
    #     k2_correct_predictions += 1

    #k=3
    if find_most_common(target_train.iloc[indices[0][0:3]]) == target_test.iloc[i]:
        k3_correct_predictions += 1
        correct_dist_k3.append(np.mean(distances[0][0:3]))
    else:
        incorrect_dist_k3.append(np.mean(distances[0][0:3]))

    #k=5
    # print()
    # print(target_train.iloc[indices[0][0]])
    # print(target_train.iloc[indices[0][0:2]])
    # print(target_test.iloc[i])
    if find_most_common(target_train.iloc[indices[0][0:5]]) == target_test.iloc[i]:
        k5_correct_predictions += 1
        correct_dist_k5.append(np.mean(distances[0][0:5]))
    else:
        incorrect_dist_k5.append(np.mean(distances[0][0:5]))


print(f"Correct predictions for TF-IDF + KNN")
print(f"Total predictions: {total_predictions}")
print(f"Accuracy for k=1: {k1_correct_predictions / total_predictions:.2f}")
# print(f"Accuracy for k=2: {k2_correct_predictions / total_predictions:.2f}")
print(f"Accuracy for k=3: {k3_correct_predictions / total_predictions:.2f}")
print(f"Accuracy for k=5: {k5_correct_predictions / total_predictions:.2f}")

print("all")
print(f"Distance stats correct predictions: Mean {np.mean(correct_dist_k1 + correct_dist_k3 + correct_dist_k5):.2f}, Max {max(correct_dist_k1 + correct_dist_k3 + correct_dist_k5):.2f}, Min {min(correct_dist_k1 + correct_dist_k3 + correct_dist_k5):.2f}")
print(f"Distance stats incorrect predictions: Mean {np.mean(incorrect_dist_k1 + incorrect_dist_k3 + incorrect_dist_k5):.2f}, Max {max(incorrect_dist_k1 + incorrect_dist_k3 + incorrect_dist_k5):.2f}, Min {min(incorrect_dist_k1 + incorrect_dist_k3 + incorrect_dist_k5):.2f}")

print("k1")
print(f"Distance stats correct predictions: Mean {np.mean(correct_dist_k1):.2f}, Max {max(correct_dist_k1):.2f}, Min {min(correct_dist_k1):.2f}")
print(f"Distance stats incorrect predictions: Mean {np.mean(incorrect_dist_k1):.2f}, Max {max(incorrect_dist_k1):.2f}, Min {min(incorrect_dist_k1):.2f}")

print("k3")
print(f"Distance stats correct predictions: Mean {np.mean(correct_dist_k3):.2f}, Max {max(correct_dist_k3):.2f}, Min {min(correct_dist_k3):.2f}")
print(f"Distance stats incorrect predictions: Mean {np.mean(incorrect_dist_k3):.2f}, Max {max(incorrect_dist_k3):.2f}, Min {min(incorrect_dist_k3):.2f}")

Correct predictions for TF-IDF + KNN
Total predictions: 386
Accuracy for k=1: 0.45
Accuracy for k=3: 0.47
Accuracy for k=5: 0.47
all
Distance stats correct predictions: Mean 0.60, Max 1.00, Min 0.24
Distance stats incorrect predictions: Mean 0.48, Max 1.00, Min 0.25
k1
Distance stats correct predictions: Mean 0.68, Max 1.00, Min 0.30
Distance stats incorrect predictions: Mean 0.53, Max 1.00, Min 0.29
k3
Distance stats correct predictions: Mean 0.59, Max 1.00, Min 0.26
Distance stats incorrect predictions: Mean 0.47, Max 1.00, Min 0.26


In [22]:
#add distance thershold

DIST_THERESHOLD_K1 = 0.65
DIST_THERESHOLD_K3 = 0.55
DIST_THERESHOLD_K5 = 0.55

total_predictions = features_test.shape[0]
k1_correct_predictions = 0
k3_correct_predictions = 0
k5_correct_predictions = 0
k1_incorrect_predictions = 0
k3_incorrect_predictions = 0
k5_incorrect_predictions = 0
k1_skiped_predictions = 0
k3_skiped_predictions = 0
k5_skiped_predictions = 0

for i in range(features_test.shape[0]):
    sample = features_test.iloc[i]
    sample_vector = vectorizer.transform([sample])
    distances, indices = knn.kneighbors(sample_vector)
    distances = 1 - distances

    #k=1
    if distances[0][0] < DIST_THERESHOLD_K1:
        k1_skiped_predictions += 1
    elif target_train.iloc[indices[0][0]] == target_test.iloc[i]:
        k1_correct_predictions += 1
    else:
        k1_incorrect_predictions += 1

    # #k=2
    # if find_most_common(target_train.iloc[indices[0][0:2]]) == target_test.iloc[i]:
    #     k2_correct_predictions += 1

    #k=3
    if np.mean(distances[0][0:3]) < DIST_THERESHOLD_K3:
        k3_skiped_predictions += 1
    elif find_most_common(target_train.iloc[indices[0][0:3]]) == target_test.iloc[i]:
        k3_correct_predictions += 1
    else:
        k3_incorrect_predictions += 1

    #k=5
    # print()
    # print(target_train.iloc[indices[0][0]])
    # print(target_train.iloc[indices[0][0:2]])
    # print(target_test.iloc[i])
    if np.mean(distances[0][0:5]) < DIST_THERESHOLD_K5:
        k5_skiped_predictions += 1
    elif find_most_common(target_train.iloc[indices[0][0:5]]) == target_test.iloc[i]:
        k5_correct_predictions += 1
    else:
        k5_incorrect_predictions += 1



print(f"Correct predictions for TF-IDF + KNN")
print(f"Total predictions: {total_predictions}")
print(f"Accuracy for k=1: \t\t{k1_correct_predictions / total_predictions:.2f}")
print(f"Skipped predictions for k=1: \t{k1_skiped_predictions/ total_predictions:.2f}")
print(f"Incorrect predictions for k=1: \t{k1_incorrect_predictions / total_predictions:.2f}\n")

print(f"Accuracy for k=3: \t\t{k3_correct_predictions / total_predictions:.2f}")
print(f"Skipped predictions for k=3: \t{k3_skiped_predictions/ total_predictions:.2f}")
print(f"Incorrect predictions for k=3: \t{k3_incorrect_predictions / total_predictions:.2f}\n")

print(f"Accuracy for k=5: \t\t{k5_correct_predictions / total_predictions:.2f}")
print(f"Skipped predictions for k=5: \t{k5_skiped_predictions/ total_predictions:.2f}")
print(f"Incorrect predictions for k=5: \t{k5_incorrect_predictions / total_predictions:.2f}\n")

Correct predictions for TF-IDF + KNN
Total predictions: 386
Accuracy for k=1: 		0.24
Skipped predictions for k=1: 	0.65
Incorrect predictions for k=1: 	0.11

Accuracy for k=3: 		0.25
Skipped predictions for k=3: 	0.65
Incorrect predictions for k=3: 	0.10

Accuracy for k=5: 		0.19
Skipped predictions for k=5: 	0.74
Incorrect predictions for k=5: 	0.07

